# Research Pipeline Explorer

Interactive notebook for browsing experiments, constraint sets, generated sentences, evaluations, and metrics from `research.db`.

In [22]:
import sys
from pathlib import Path

# Ensure the project root is on the path so `research.*` imports work
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from sqlalchemy import func
from research.db.database import SessionLocal, init_db
from research.db.models import (
    Benchmark,
    ConstraintSet,
    Experiment,
    ExperimentMetric,
    GeneratedSentence,
    MethodConfig,
    SentenceEvaluation,
)

init_db()
session = SessionLocal()
print(f"Connected to: {session.bind.url}")

Connected to: sqlite+pysqlite:////Users/joshuagraham/Desktop/Diss/LinguistOS/research/research.db


## 1. Experiments

In [23]:
experiments = (
    session.query(Experiment)
    .order_by(Experiment.created_at.desc())
    .all()
)

df_exp = pd.DataFrame([
    {
        "id": e.id,
        "name": e.name,
        "benchmark": e.benchmark.name if e.benchmark else None,
        "method_config": e.method_config.name if e.method_config else None,
        "generator": e.method_config.method if e.method_config else None,
        "samples_per_case": e.method_config.samples_per_case if e.method_config else None,
        "status": e.status,
        "created_at": e.created_at,
        "completed_at": e.completed_at,
    }
    for e in experiments
])
df_exp

,id,name,method,samples_per_case,status,created_at,completed_at,config
0,3,baseline_gpt_live,baseline_gpt,3,completed,2026-05-14 09:28:59.463791,2026-05-14 09:29:10.491680,"{'live': True, 'model': 'gpt-4o', 'temperature..."
1,2,baseline_gpt_mock,baseline_gpt,3,completed,2026-05-14 09:21:29.529264,2026-05-14 09:21:29.548582,"{'live': False, 'model': 'gpt-4o', 'temperatur..."
2,1,baseline_gpt_mock,baseline_gpt,3,completed,2026-05-13 17:29:49.408944,2026-05-13 17:29:49.415082,"{'live': False, 'model': 'gpt-4o', 'temperatur..."


## 2. Constraint Sets

In [24]:
constraint_sets = session.query(ConstraintSet).all()

df_cs = pd.DataFrame([
    {
        "id": cs.id,
        "keyword": cs.keyword,
        "translation": cs.translation,
        "tense": cs.tense,
        "person": cs.person,
        "number": cs.number,
        "target_language": cs.target_language,
        "cefr_level": cs.cefr_level,
    }
    for cs in constraint_sets
])
df_cs

,id,keyword,translation,tense,person,number,target_language,cefr_level
0,1,comer,to eat,past,1st,plural,es,None
1,2,vivir,to live,future,3rd,singular,es,None
2,3,hablar,to speak,present,2nd,singular,es,None
3,4,escribir,to write,past,3rd,plural,es,None
4,5,correr,to run,present,1st,singular,es,None


## 3. Generated Sentences

Pick an experiment to inspect (change `EXP_ID`):

In [25]:
EXP_ID = 3  # <-- change this to explore different experiments

sentences = (
    session.query(GeneratedSentence)
    .filter_by(experiment_id=EXP_ID)
    .order_by(GeneratedSentence.constraint_set_id, GeneratedSentence.sample_index)
    .all()
)

df_sent = pd.DataFrame([
    {
        "id": s.id,
        "constraint_set": f"{s.constraint_set.keyword} + {s.constraint_set.tense}",
        "sample": s.sample_index,
        "sentence": s.sentence,
        "translation": s.translation,
    }
    for s in sentences
])

print(f"Experiment {EXP_ID}: {len(df_sent)} sentences")
df_sent

Experiment 3: 15 sentences


,id,constraint_set,sample,sentence,translation
0,31,comer + past,0,Comimos pizza anoche.,We ate pizza last night.
1,32,comer + past,1,Ayer comimos temprano.,Yesterday we ate early.
2,33,comer + past,2,Comimos juntos en casa.,We ate together at home.
3,34,vivir + future,0,Él vivirá aquí.,He will live here.
4,35,vivir + future,1,Ella vivirá feliz.,She will live happily.
5,36,vivir + future,2,Vivirá en Madrid.,She/He will live in Madrid.
6,37,hablar + present,0,Hablas muy bien.,You speak very well.
7,38,hablar + present,1,¿Hablas español?,Do you speak Spanish?
8,39,hablar + present,2,Hablas demasiado.,You speak too much.
9,40,escribir + past,0,Ellos escribieron cartas.,They wrote letters.


## 4. Sentence Evaluations (Stage 1)

Per-sentence scores from all evaluators:

In [26]:
evals = (
    session.query(
        SentenceEvaluation.id,
        GeneratedSentence.constraint_set_id,
        SentenceEvaluation.evaluator_name,
        SentenceEvaluation.score,
        GeneratedSentence.sentence,
        SentenceEvaluation.details,
    )
    .join(GeneratedSentence, SentenceEvaluation.sentence_id == GeneratedSentence.id)
    .filter(GeneratedSentence.experiment_id == EXP_ID)
    .order_by(SentenceEvaluation.evaluator_name, GeneratedSentence.constraint_set_id)
    .all()
)

df_evals = pd.DataFrame(evals, columns=["eval_id", "cs_id", "evaluator", "score", "sentence", "details"])
print(f"{len(df_evals)} evaluation rows")
df_evals

15 evaluation rows


,eval_id,cs_id,evaluator,score,sentence,details
0,16,1,grammar_stub,1.0,Comimos pizza anoche.,"{'has_keyword_stem': True, 'has_translation': ..."
1,17,1,grammar_stub,1.0,Ayer comimos temprano.,"{'has_keyword_stem': True, 'has_translation': ..."
2,18,1,grammar_stub,1.0,Comimos juntos en casa.,"{'has_keyword_stem': True, 'has_translation': ..."
3,19,2,grammar_stub,1.0,Él vivirá aquí.,"{'has_keyword_stem': True, 'has_translation': ..."
4,20,2,grammar_stub,1.0,Ella vivirá feliz.,"{'has_keyword_stem': True, 'has_translation': ..."
5,21,2,grammar_stub,1.0,Vivirá en Madrid.,"{'has_keyword_stem': True, 'has_translation': ..."
6,22,3,grammar_stub,1.0,Hablas muy bien.,"{'has_keyword_stem': True, 'has_translation': ..."
7,23,3,grammar_stub,1.0,¿Hablas español?,"{'has_keyword_stem': True, 'has_translation': ..."
8,24,3,grammar_stub,1.0,Hablas demasiado.,"{'has_keyword_stem': True, 'has_translation': ..."
9,25,4,grammar_stub,1.0,Ellos escribieron cartas.,"{'has_keyword_stem': True, 'has_translation': ..."


In [27]:
# Score distribution per evaluator
if not df_evals.empty:
    print(df_evals.groupby("evaluator")["score"].describe().round(4))

              count  mean  std  min  25%  50%  75%  max
evaluator                                              
grammar_stub   15.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0


## 5. Experiment Metrics (Stage 2a roll-ups + Stage 2b distribution)

In [28]:
metrics = (
    session.query(ExperimentMetric)
    .filter_by(experiment_id=EXP_ID)
    .order_by(ExperimentMetric.scope, ExperimentMetric.metric_name)
    .all()
)

df_metrics = pd.DataFrame([
    {
        "metric_name": m.metric_name,
        "value": round(m.value, 4),
        "scope": m.scope,
        "constraint_set_id": m.constraint_set_id,
        "breakdown": m.breakdown,
    }
    for m in metrics
])

print(f"{len(df_metrics)} metric rows")
df_metrics

12 metric rows


,metric_name,value,scope,constraint_set_id,breakdown
0,mean::grammar_stub,1.0,constraint_set,1.0,"{'evaluator': 'grammar_stub', 'count': 3}"
1,mean::grammar_stub,1.0,constraint_set,2.0,"{'evaluator': 'grammar_stub', 'count': 3}"
2,mean::grammar_stub,1.0,constraint_set,3.0,"{'evaluator': 'grammar_stub', 'count': 3}"
3,mean::grammar_stub,1.0,constraint_set,4.0,"{'evaluator': 'grammar_stub', 'count': 3}"
4,mean::grammar_stub,1.0,constraint_set,5.0,"{'evaluator': 'grammar_stub', 'count': 3}"
5,uniqueness_ratio,1.0,constraint_set,1.0,"{'unique': 3, 'n': 3}"
6,uniqueness_ratio,1.0,constraint_set,2.0,"{'unique': 3, 'n': 3}"
7,uniqueness_ratio,1.0,constraint_set,3.0,"{'unique': 3, 'n': 3}"
8,uniqueness_ratio,1.0,constraint_set,4.0,"{'unique': 3, 'n': 3}"
9,uniqueness_ratio,1.0,constraint_set,5.0,"{'unique': 3, 'n': 3}"


In [29]:
# Experiment-wide summary (one row per metric)
if not df_metrics.empty:
    exp_wide = df_metrics[df_metrics["scope"] == "experiment"][["metric_name", "value"]]
    print("\nExperiment-wide metrics:")
    print(exp_wide.to_string(index=False))


Experiment-wide metrics:
                metric_name  value
         mean::grammar_stub    1.0
uniqueness_ratio_experiment    1.0


In [30]:
# Per-constraint-set heatmap of metrics
if not df_metrics.empty:
    cs_metrics = df_metrics[df_metrics["scope"] == "constraint_set"].copy()
    if not cs_metrics.empty:
        pivot = cs_metrics.pivot_table(
            index="constraint_set_id", columns="metric_name", values="value"
        )
        styled = pivot.style.background_gradient(cmap="RdYlGn", axis=None).format("{:.4f}")
        display(styled)

metric_name,mean::grammar_stub,uniqueness_ratio
constraint_set_id,,
1.000000,1.0000,1.0000
2.000000,1.0000,1.0000
3.000000,1.0000,1.0000
4.000000,1.0000,1.0000
5.000000,1.0000,1.0000


## 6. Compare experiments

Side-by-side experiment-wide metrics (roll-ups and distribution). Change `EXP_IDS` to the experiment ids you want to compare.

In [ ]:
EXP_IDS = [1, 2]  # <-- experiment ids to compare

metrics = (
    session.query(ExperimentMetric, Experiment.name)
    .join(Experiment, ExperimentMetric.experiment_id == Experiment.id)
    .filter(ExperimentMetric.experiment_id.in_(EXP_IDS))
    .filter(ExperimentMetric.scope == "experiment")
    .all()
)

rows = [
    {
        "experiment_id": m.experiment_id,
        "experiment": name,
        "metric_name": m.metric_name,
        "value": round(m.value, 4),
    }
    for m, name in metrics
]

df_compare = pd.DataFrame(rows)
if df_compare.empty:
    print("No experiment-wide metrics for those ids.")
else:
    pivot = df_compare.pivot_table(
        index="metric_name",
        columns="experiment",
        values="value",
    )
    display(pivot.style.background_gradient(cmap="RdYlGn", axis=None).format("{:.4f}"))

## 7. Quick counts

In [31]:
print(f"Experiments:          {session.query(Experiment).count()}")
print(f"Constraint sets:      {session.query(ConstraintSet).count()}")
print(f"Generated sentences:  {session.query(GeneratedSentence).count()}")
print(f"Sentence evaluations: {session.query(SentenceEvaluation).count()}")
print(f"Experiment metrics:   {session.query(ExperimentMetric).count()}")

Experiments:          3
Constraint sets:      5
Generated sentences:  45
Sentence evaluations: 30
Experiment metrics:   24


In [32]:
session.close()